# Tie-out for notebooks 13–25 (A21)

`10_tie_out` checks `00`–`09`. This notebook gives `13`–`25` the same guarantee and prints, in one place, every
number the paper cites from them. It trains nothing.

Checks: (1) provenance of each result and of the notebooks it depends on; (2) stored hyperparameters equal `02`'s
(and `17`'s Random Forest equals `13`'s); (3) `14` ran on predictions that reproduce the committed `03`/`04`/`06`;
(4) every result records its platform. All checks are collected, the results are saved, and the last cell fails if
any check failed.

Numbered 99 so that it runs last under the README's glob (it was `21_tie_out_revision`).

**Results**: `results/99_tie_out_revision.json`.

In [1]:
import json, os, subprocess, warnings
import nbformat
import pandas as pd
import sybil_pipeline as sp
warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
assert not sp.CODE_COMMIT.endswith('-dirty'), f'Run from a clean working tree (CODE_COMMIT={sp.CODE_COMMIT})'
print('Code commit:', sp.CODE_COMMIT)
NBS = [nb for nb in ['13_random_forest_sybil', '14_entity_level_recall', '15_split_tree_report',
                     '16_label_robustness_initial_list', '17_models_10_splits', '18_mimicry_stress',
                     '19_ablation_by_category', '20_temporal_holdout', '22_report_dependence', '23_revision_figures', '24_mimicry_controls', '25_paper_tables'] if os.path.exists(f'results/{nb}.json')]
R = {nb: sp.load_results(nb) for nb in NBS}
print('Results found:', NBS)
CHECKS = []
def check(name, ok, detail=''):
    CHECKS.append(dict(check=name, passed=bool(ok), detail=detail))
    print(f"{'PASS' if ok else 'FAIL'}  {name}" + (f'  ({detail})' if detail else ''))

Code commit: 5b6e441
Results found: ['13_random_forest_sybil', '14_entity_level_recall', '15_split_tree_report', '16_label_robustness_initial_list', '17_models_10_splits', '18_mimicry_stress', '19_ablation_by_category', '20_temporal_holdout', '22_report_dependence', '23_revision_figures', '24_mimicry_controls', '25_paper_tables']


## 1. Provenance, with explicit dependencies

In [2]:
DEPS = {'13_random_forest_sybil': ['02_hyperparameter_search', '08_ablation_families'],
        '14_entity_level_recall': ['03_xgboost_sybil', '04_lightgbm_sybil', '06_cross_ensemble_sybil'],
        '15_split_tree_report': ['02_hyperparameter_search', '03_xgboost_sybil', '04_lightgbm_sybil', '08_ablation_families'],
        '16_label_robustness_initial_list': ['02_hyperparameter_search'],
        '17_models_10_splits': ['02_hyperparameter_search', '08_ablation_families', '13_random_forest_sybil'],
        '18_mimicry_stress': ['02_hyperparameter_search', '09_shap_importance'],
        '19_ablation_by_category': ['02_hyperparameter_search', '08_ablation_families'],
        '20_temporal_holdout': ['02_hyperparameter_search'],
        '22_report_dependence': ['02_hyperparameter_search', '15_split_tree_report'],
        '23_revision_figures': ['03_xgboost_sybil', '04_lightgbm_sybil', '09_shap_importance', '11_split_comparison',
                                '16_label_robustness_initial_list', '17_models_10_splits', '18_mimicry_stress',
                                '22_report_dependence'],
        '24_mimicry_controls': ['02_hyperparameter_search', '09_shap_importance', '18_mimicry_stress'],
        '25_paper_tables': ['00_data_pipeline', '02_hyperparameter_search', '07_sensitivity_label_vintage', '08_ablation_families',
                            '11_split_comparison', '13_random_forest_sybil', '14_entity_level_recall', '15_split_tree_report',
                            '16_label_robustness_initial_list', '17_models_10_splits', '19_ablation_by_category',
                            '20_temporal_holdout', '22_report_dependence']}
git = lambda *a: subprocess.run(['git', *a], capture_output=True, text=True)
def code_cells(ref, nb):
    text = git('show', f'{ref}:{nb}.ipynb').stdout
    return [c.source for c in nbformat.reads(text, as_version=4).cells if c.cell_type == 'code'] if text else None

prov = sp.provenance(NBS)
print(prov.to_string(index=False))
for r in prov.itertuples():
    check(f'provenance {r.notebook}', r.dependencies_unchanged, f'commit {r.commit}')
for nb in NBS:
    commit = R[nb]['code_commit']
    for dep in DEPS[nb]:
        same_code = code_cells(commit, dep) == code_cells('HEAD', dep)
        dep_ok = bool(sp.provenance([dep])['dependencies_unchanged'].all())
        check(f'{nb} -> {dep}', same_code and dep_ok,
              f"code of {dep} unchanged since {commit}: {same_code}; {dep}'s own result traces to HEAD: {dep_ok}")

                        notebook  commit  dependencies_unchanged
          13_random_forest_sybil 073ecca                    True
          14_entity_level_recall 4f2d3a9                    True
            15_split_tree_report 9753ef1                    True
16_label_robustness_initial_list a03cf26                    True
             17_models_10_splits 8d24f2e                    True
               18_mimicry_stress 5ab0bb1                    True
         19_ablation_by_category 5ceb757                    True
             20_temporal_holdout 31d466b                    True
            22_report_dependence 7da3628                    True
             23_revision_figures 3b1c175                    True
             24_mimicry_controls 693757c                    True
                 25_paper_tables d31479e                    True
PASS  provenance 13_random_forest_sybil  (commit 073ecca)
PASS  provenance 14_entity_level_recall  (commit 4f2d3a9)
PASS  provenance 15_split_tree_report  

PASS  14_entity_level_recall -> 06_cross_ensemble_sybil  (code of 06_cross_ensemble_sybil unchanged since 4f2d3a9: True; 06_cross_ensemble_sybil's own result traces to HEAD: True)
PASS  15_split_tree_report -> 02_hyperparameter_search  (code of 02_hyperparameter_search unchanged since 9753ef1: True; 02_hyperparameter_search's own result traces to HEAD: True)
PASS  15_split_tree_report -> 03_xgboost_sybil  (code of 03_xgboost_sybil unchanged since 9753ef1: True; 03_xgboost_sybil's own result traces to HEAD: True)
PASS  15_split_tree_report -> 04_lightgbm_sybil  (code of 04_lightgbm_sybil unchanged since 9753ef1: True; 04_lightgbm_sybil's own result traces to HEAD: True)
PASS  15_split_tree_report -> 08_ablation_families  (code of 08_ablation_families unchanged since 9753ef1: True; 08_ablation_families's own result traces to HEAD: True)
PASS  16_label_robustness_initial_list -> 02_hyperparameter_search  (code of 02_hyperparameter_search unchanged since a03cf26: True; 02_hyperparameter_se

PASS  17_models_10_splits -> 02_hyperparameter_search  (code of 02_hyperparameter_search unchanged since 8d24f2e: True; 02_hyperparameter_search's own result traces to HEAD: True)
PASS  17_models_10_splits -> 08_ablation_families  (code of 08_ablation_families unchanged since 8d24f2e: True; 08_ablation_families's own result traces to HEAD: True)
PASS  17_models_10_splits -> 13_random_forest_sybil  (code of 13_random_forest_sybil unchanged since 8d24f2e: True; 13_random_forest_sybil's own result traces to HEAD: True)
PASS  18_mimicry_stress -> 02_hyperparameter_search  (code of 02_hyperparameter_search unchanged since 5ab0bb1: True; 02_hyperparameter_search's own result traces to HEAD: True)
PASS  18_mimicry_stress -> 09_shap_importance  (code of 09_shap_importance unchanged since 5ab0bb1: True; 09_shap_importance's own result traces to HEAD: True)
PASS  19_ablation_by_category -> 02_hyperparameter_search  (code of 02_hyperparameter_search unchanged since 5ceb757: True; 02_hyperparamete

PASS  19_ablation_by_category -> 08_ablation_families  (code of 08_ablation_families unchanged since 5ceb757: True; 08_ablation_families's own result traces to HEAD: True)
PASS  20_temporal_holdout -> 02_hyperparameter_search  (code of 02_hyperparameter_search unchanged since 31d466b: True; 02_hyperparameter_search's own result traces to HEAD: True)
PASS  22_report_dependence -> 02_hyperparameter_search  (code of 02_hyperparameter_search unchanged since 7da3628: True; 02_hyperparameter_search's own result traces to HEAD: True)
PASS  22_report_dependence -> 15_split_tree_report  (code of 15_split_tree_report unchanged since 7da3628: True; 15_split_tree_report's own result traces to HEAD: True)
PASS  23_revision_figures -> 03_xgboost_sybil  (code of 03_xgboost_sybil unchanged since 3b1c175: True; 03_xgboost_sybil's own result traces to HEAD: True)
PASS  23_revision_figures -> 04_lightgbm_sybil  (code of 04_lightgbm_sybil unchanged since 3b1c175: True; 04_lightgbm_sybil's own result trace

PASS  23_revision_figures -> 09_shap_importance  (code of 09_shap_importance unchanged since 3b1c175: True; 09_shap_importance's own result traces to HEAD: True)
PASS  23_revision_figures -> 11_split_comparison  (code of 11_split_comparison unchanged since 3b1c175: True; 11_split_comparison's own result traces to HEAD: True)
PASS  23_revision_figures -> 16_label_robustness_initial_list  (code of 16_label_robustness_initial_list unchanged since 3b1c175: True; 16_label_robustness_initial_list's own result traces to HEAD: True)
PASS  23_revision_figures -> 17_models_10_splits  (code of 17_models_10_splits unchanged since 3b1c175: True; 17_models_10_splits's own result traces to HEAD: True)
PASS  23_revision_figures -> 18_mimicry_stress  (code of 18_mimicry_stress unchanged since 3b1c175: True; 18_mimicry_stress's own result traces to HEAD: True)
PASS  23_revision_figures -> 22_report_dependence  (code of 22_report_dependence unchanged since 3b1c175: True; 22_report_dependence's own result

PASS  24_mimicry_controls -> 09_shap_importance  (code of 09_shap_importance unchanged since 693757c: True; 09_shap_importance's own result traces to HEAD: True)
PASS  24_mimicry_controls -> 18_mimicry_stress  (code of 18_mimicry_stress unchanged since 693757c: True; 18_mimicry_stress's own result traces to HEAD: True)
PASS  25_paper_tables -> 00_data_pipeline  (code of 00_data_pipeline unchanged since d31479e: True; 00_data_pipeline's own result traces to HEAD: True)
PASS  25_paper_tables -> 02_hyperparameter_search  (code of 02_hyperparameter_search unchanged since d31479e: True; 02_hyperparameter_search's own result traces to HEAD: True)
PASS  25_paper_tables -> 07_sensitivity_label_vintage  (code of 07_sensitivity_label_vintage unchanged since d31479e: True; 07_sensitivity_label_vintage's own result traces to HEAD: True)
PASS  25_paper_tables -> 08_ablation_families  (code of 08_ablation_families unchanged since d31479e: True; 08_ablation_families's own result traces to HEAD: True)

PASS  25_paper_tables -> 11_split_comparison  (code of 11_split_comparison unchanged since d31479e: True; 11_split_comparison's own result traces to HEAD: True)
PASS  25_paper_tables -> 13_random_forest_sybil  (code of 13_random_forest_sybil unchanged since d31479e: True; 13_random_forest_sybil's own result traces to HEAD: True)
PASS  25_paper_tables -> 14_entity_level_recall  (code of 14_entity_level_recall unchanged since d31479e: True; 14_entity_level_recall's own result traces to HEAD: True)
PASS  25_paper_tables -> 15_split_tree_report  (code of 15_split_tree_report unchanged since d31479e: True; 15_split_tree_report's own result traces to HEAD: True)
PASS  25_paper_tables -> 16_label_robustness_initial_list  (code of 16_label_robustness_initial_list unchanged since d31479e: True; 16_label_robustness_initial_list's own result traces to HEAD: True)
PASS  25_paper_tables -> 17_models_10_splits  (code of 17_models_10_splits unchanged since d31479e: True; 17_models_10_splits's own res

PASS  25_paper_tables -> 20_temporal_holdout  (code of 20_temporal_holdout unchanged since d31479e: True; 20_temporal_holdout's own result traces to HEAD: True)
PASS  25_paper_tables -> 22_report_dependence  (code of 22_report_dependence unchanged since d31479e: True; 22_report_dependence's own result traces to HEAD: True)


## 2. Hyperparameters

In [3]:
XGB_P, LGBM_P = sp.load_selected_params()
stored = {'13_random_forest_sybil': ('lgbm_params', LGBM_P), '16_label_robustness_initial_list': ('params', LGBM_P),
          '18_mimicry_stress': ('params', LGBM_P), '19_ablation_by_category': ('params', LGBM_P),
          '20_temporal_holdout': ('params', LGBM_P)}
for nb, (key, ref) in stored.items():
    if nb in R:
        check(f'{nb} LightGBM parameters = 02', R[nb].get(key) == ref, str(R[nb].get(key)))
if '17_models_10_splits' in R:
    p = R['17_models_10_splits']['params']
    check('17 XGBoost parameters = 02', p['xgb'] == XGB_P); check('17 LightGBM parameters = 02', p['lgbm'] == LGBM_P)
    check('17 Random Forest parameters = 13', p['rf'] == R['13_random_forest_sybil']['params'], str(p['rf']))
if '25_paper_tables' in R:
    tn = json.load(open('tables/rev_text_numbers.json'))
    check('25 text numbers: no missing names', not tn['missing'] and not R['25_paper_tables']['text_numbers_missing'],
          f"{len(tn['numbers'])} names; missing {tn['missing']}")
if '24_mimicry_controls' in R:
    check('24 LightGBM parameters = 02', R['24_mimicry_controls']['params'] == LGBM_P)
    rc = R['24_mimicry_controls']['regression_check']
    ok = all(abs(v['here'] - v['r18']) < 1e-9 for v in rc['baseline'].values()) and all(abs(x['diff']) < 1e-9 for x in rc['recall'])
    check('24 reproduces 18 (baseline and recall regression checks)', ok, f"{len(rc['recall'])} recall checks")
if '22_report_dependence' in R:
    p = R['22_report_dependence']['params']
    check('22 LightGBM parameters = 02', p['lgbm'] == LGBM_P); check('22 XGBoost parameters = 02', p['xgb'] == XGB_P)
if '15_split_tree_report' in R:
    check('15 uses 02 (loaded at run time; no parameters stored)', 'from 02' in R['15_split_tree_report']['hyperparameters'])

PASS  13_random_forest_sybil LightGBM parameters = 02  ({'num_leaves': 127, 'learning_rate': 0.05, 'subsample': 0.7, 'reg_lambda': 1.0})
PASS  16_label_robustness_initial_list LightGBM parameters = 02  ({'num_leaves': 127, 'learning_rate': 0.05, 'subsample': 0.7, 'reg_lambda': 1.0})
PASS  18_mimicry_stress LightGBM parameters = 02  ({'num_leaves': 127, 'learning_rate': 0.05, 'subsample': 0.7, 'reg_lambda': 1.0})
PASS  19_ablation_by_category LightGBM parameters = 02  ({'num_leaves': 127, 'learning_rate': 0.05, 'subsample': 0.7, 'reg_lambda': 1.0})
PASS  20_temporal_holdout LightGBM parameters = 02  ({'num_leaves': 127, 'learning_rate': 0.05, 'subsample': 0.7, 'reg_lambda': 1.0})
PASS  17 XGBoost parameters = 02
PASS  17 LightGBM parameters = 02
PASS  17 Random Forest parameters = 13  ({'max_features': 0.3, 'min_samples_leaf': 1, 'max_depth': None, 'max_samples': None, 'n_estimators': 300, 'criterion': 'gini', 'bootstrap': True, 'n_jobs': 4})
PASS  25 text numbers: no missing names  (19

## 3. Inputs of `14`

In [4]:
if '14_entity_level_recall' in R:
    ok = R['14_entity_level_recall']['inputs']['predictions_match_committed_results']
    check('14 predictions reproduce the committed 03/04/06', ok,
          'OK' if ok else 'rerun 03, 04, 06 and then 14 on the machine that produced the committed results (about 15 min)')

PASS  14 predictions reproduce the committed 03/04/06  (OK)


## 4. Platform

In [5]:
plat = {}
for nb in NBS:
    if nb == '25_paper_tables':       # trains nothing; records its own run platform and each source's platform
        check('25 records the platform of each source', 'platform' in R[nb] and bool(R[nb]['platforms']))
        continue
    if nb == '23_revision_figures':   # trains nothing; records each source's platform instead
        check('23 records the platform of each source', all('platform' in s for v in R[nb]['sources'].values() for s in v))
        continue
    p = R[nb].get('platform') or R[nb].get('inputs', {}).get('platform')
    check(f'{nb} records its platform', p is not None)
    if p:
        plat[nb] = {k: p.get(k) for k in ['machine', 'system', 'processor', 'python', 'cpu_count', 'n_jobs']}
plat = pd.DataFrame(plat).T
print(plat.to_string())
machines = set(map(tuple, plat[['machine', 'system']].values))
# 00-12 record no platform; 14 shows their machine differs from arm64/Darwin (03/04 do not reproduce there).
other_00_12 = not R.get('14_entity_level_recall', {}).get('inputs', {}).get('predictions_match_committed_results', True)
print(f'\nPlatforms among 13-25: {sorted(machines)}; 00-12: not recorded'
      + (', and not the platform of 14 (its rerun of 03/04 did not reproduce them).' if other_00_12 else '.'))
MULTI_PLATFORM = len(machines) > 1 or other_00_12
if MULTI_PLATFORM:
    print('WARNING: the paper would cite numbers from more than one platform (00-12 and 13-25). '
          'Comparisons within 13-25 are same-machine; comparisons with 00-12 differ by about 0.002 F1 across platforms.')

PASS  13_random_forest_sybil records its platform
PASS  14_entity_level_recall records its platform
PASS  15_split_tree_report records its platform
PASS  16_label_robustness_initial_list records its platform
PASS  17_models_10_splits records its platform
PASS  18_mimicry_stress records its platform
PASS  19_ablation_by_category records its platform
PASS  20_temporal_holdout records its platform
PASS  22_report_dependence records its platform
PASS  23 records the platform of each source
PASS  24_mimicry_controls records its platform
PASS  25 records the platform of each source
                                 machine system processor   python cpu_count n_jobs
13_random_forest_sybil            x86_64  Linux    x86_64  3.11.15         4      4
14_entity_level_recall            x86_64  Linux    x86_64  3.11.15       NaN    NaN
15_split_tree_report              x86_64  Linux    x86_64  3.11.15         4      4
16_label_robustness_initial_list  x86_64  Linux    x86_64  3.11.15         4     

## 5. Numbers cited from 13–20

In [6]:
CITED = {}
def show(title, obj):
    CITED[title] = obj
    print(f'=== {title}'); print(json.dumps(obj, indent=1, default=float)[:4000]); print()

if '17_models_10_splits' in R:
    show('Model table: 10 group splits, mean ± SD, p vs LightGBM (17)', R['17_models_10_splits']['cite_table'])
if '13_random_forest_sybil' in R:
    r = R['13_random_forest_sybil']
    show('Random Forest (13)', dict(selected=r['params'], test=r['models'][0],
                                     ten_splits={k: v for k, v in r['summary'].items() if k.startswith(('test_f1', 'lgbm_test_f1'))}))
if '14_entity_level_recall' in R:
    t = pd.DataFrame(R['14_entity_level_recall']['table'])
    t = t[(t.threshold_label == 'validation') & (t.bucket == 'all') & t.model.isin(['LightGBM', 'Ensemble'])]
    show('Entity-level recall (14)', t[['entity', 'model', 'entities', 'any_hit', 'majority_hit', 'full_hit',
                                        'residual_allocation_share', 'entities_majority_allocation_unflagged']].to_dict('records'))
if '15_split_tree_report' in R:
    show('Report-level dependence (15)', R['15_split_tree_report']['interpretation_inputs'])
if '16_label_robustness_initial_list' in R:
    r = R['16_label_robustness_initial_list']
    show('Initial-list robustness (16)', dict(counts=r['counts'], fp_at_validation_threshold=[x for x in r['fp_composition'] if x['note']],
                                              table_2x2=r['table_2x2'], variant_B=r['variant_B'],
                                              ten_splits={k: v for k, v in r['summary'].items() if k.startswith('delta_')}))
if '18_mimicry_stress' in R:
    r = R['18_mimicry_stress']
    show('Mimicry stress test (18)', dict(baseline=r['baseline'], k5_k10=[{k: x[k] for k in ['features', 'attack', 'k', 'recall_mean', 'recall_std']}
                                                                       for x in r['results'] if x['k'] in (5, 10)]))
if '19_ablation_by_category' in R:
    d = pd.DataFrame(R['19_ablation_by_category']['diffs'])
    show('Category ablation (19)', d[d.metric.isin(['f1', 'recall'])][['without', 'category', 'metric', 'diff_mean', 'diff_sd', 'all_higher_n', 'p_corrected']].to_dict('records'))
if '24_mimicry_controls' in R:
    r = R['24_mimicry_controls']
    show('Mimicry controls (24)', dict(shap_shares=r['shap_shares'], single_feature_top5=r['single_feature'][:5],
                                       tie_check=r['single_feature_tie_check']))
if '20_temporal_holdout' in R:
    show('Temporal holdout (20)', R['20_temporal_holdout']['results'])
if '22_report_dependence' in R:
    r = R['22_report_dependence']
    show('Report dependence: seen minus held out (22)', dict(tests=r['tests'], dose_response=r['dose_response'],
                                                            composition=r['composition_summary']))

=== Model table: 10 group splits, mean ± SD, p vs LightGBM (17)
[
 {
  "model": "Logistic regression",
  "F1": "0.2336 \u00b1 0.0041",
  "p F1": "8.71e-14",
  "AP": "0.1605 \u00b1 0.0088",
  "p AP": "8.55e-13",
  "AUROC": "0.8323 \u00b1 0.0030",
  "p AUROC": "9.43e-12"
 },
 {
  "model": "XGBoost",
  "F1": "0.7066 \u00b1 0.0076",
  "p F1": "0.457",
  "AP": "0.7529 \u00b1 0.0085",
  "p AP": "0.0834",
  "AUROC": "0.9641 \u00b1 0.0020",
  "p AUROC": "0.0302"
 },
 {
  "model": "LightGBM",
  "F1": "0.7131 \u00b1 0.0060",
  "p F1": "\u2014",
  "AP": "0.7607 \u00b1 0.0060",
  "p AP": "\u2014",
  "AUROC": "0.9682 \u00b1 0.0018",
  "p AUROC": "\u2014"
 },
 {
  "model": "Random Forest",
  "F1": "0.7022 \u00b1 0.0056",
  "p F1": "0.32",
  "AP": "0.7535 \u00b1 0.0073",
  "p AP": "0.0834",
  "AUROC": "0.9685 \u00b1 0.0015",
  "p AUROC": "1"
 },
 {
  "model": "Ensemble",
  "F1": "0.7128 \u00b1 0.0082",
  "p F1": "0.941",
  "AP": "0.7610 \u00b1 0.0070",
  "p AP": "0.86",
  "AUROC": "0.9680 \u00b1 0.00

## Save, then fail if any check failed

In [7]:
checks = pd.DataFrame(CHECKS)
failed = checks[~checks['passed']]
sp.save_results([], '99_tie_out_revision', extra=dict(checks=CHECKS, all_passed=bool(checks['passed'].all()),
                                                      multiple_platforms_cited=MULTI_PLATFORM,
                                                      platforms=plat.reset_index().to_dict('records'), cited=CITED))
print(f"{int(checks['passed'].sum())} of {len(checks)} checks passed")
print(failed.to_string(index=False) if len(failed) else 'All checks passed')

Saved results/99_tie_out_revision.json
83 of 83 checks passed
All checks passed


In [8]:
assert failed.empty, f'{len(failed)} check(s) failed: ' + '; '.join(failed['check'] + ': ' + failed['detail'])